# 05 · Statistical vs Practical Significance

## What this notebook covers
A p-value tells you whether an effect is real. It does not tell you whether it matters. This notebook operationalises the distinction using effect sizes, power analysis, and business impact framing.

## Why the distinction matters
With n=100,000 samples, even a difference of ΔAUC=0.0001 will produce p<0.05. The model is meaningfully different in a mathematical sense but identical in practice. Conversely, with n=100 samples, a real and economically important ΔAUC=0.05 may not reach significance — the study was underpowered.

## Tools
- **Cohen's d** — standardised mean difference for continuous outcomes
- **Cohen's h** — effect size for proportions / AUC comparison
- **Power analysis** — compute minimum detectable effect (MDE) for a given n, or required n for a given MDE
- **Business impact framing** — convert ΔAUC to revenue / cost implications

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from scipy import stats
from statsmodels.stats.power import TTestPower
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

def cohens_d(x, y):
    """Pooled-SD Cohen's d for paired samples."""
    diff = np.array(x) - np.array(y)
    return diff.mean() / diff.std(ddof=1)

def cohens_h(auc_a, auc_b):
    """Cohen's h for comparing two AUC values treated as proportions."""
    phi_a = 2 * np.arcsin(np.sqrt(auc_a))
    phi_b = 2 * np.arcsin(np.sqrt(auc_b))
    return abs(phi_a - phi_b)

X, y = make_classification(n_samples=2000, n_features=20, n_informative=10, random_state=0)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

gbm = GradientBoostingClassifier(n_estimators=100, random_state=0)
rf  = RandomForestClassifier(n_estimators=100, random_state=0)
auc_gbm = cross_val_score(gbm, X, y, cv=cv, scoring="roc_auc")
auc_rf  = cross_val_score(rf,  X, y, cv=cv, scoring="roc_auc")

d = cohens_d(auc_gbm, auc_rf)
h = cohens_h(auc_gbm.mean(), auc_rf.mean())
_, p_val = stats.wilcoxon(auc_gbm, auc_rf)

print(f"GBM mean AUC : {auc_gbm.mean():.4f}")
print(f"RF  mean AUC : {auc_rf.mean():.4f}")
print(f"Delta        : {auc_gbm.mean() - auc_rf.mean():.4f}")
print(f"Cohen's d    : {d:.3f}  ({'negligible' if abs(d)<0.2 else 'small' if abs(d)<0.5 else 'medium' if abs(d)<0.8 else 'large'})")
print(f"Cohen's h    : {h:.3f}  ({'negligible' if h<0.2 else 'small' if h<0.5 else 'medium' if h<0.8 else 'large'})")
print(f"p-value      : {p_val:.4f}")

In [ ]:
# Power analysis: minimum required test-set size for various MDEs
power_analysis = TTestPower()
alphas = [0.05]
target_power = 0.80
mdes = np.linspace(0.01, 0.10, 50)

required_n = [
    power_analysis.solve_power(effect_size=mde, alpha=0.05, power=target_power, alternative="two-sided")
    for mde in mdes
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(mdes, required_n, color="steelblue", lw=2)
axes[0].axhline(len(y) * 0.3, color="red", linestyle="--", label=f"Our test set (n={int(len(y)*0.3)})")
axes[0].set_xlabel("Minimum Detectable Effect (ΔAUC)")
axes[0].set_ylabel("Required test-set size")
axes[0].set_title("Power analysis: n needed vs MDE (power=0.80, α=0.05)")
axes[0].legend()

# Power curve: power vs test-set size for fixed MDE
n_range = np.linspace(50, 1000, 100)
powers_small = [power_analysis.solve_power(effect_size=0.02, alpha=0.05, nobs=n, alternative="two-sided") for n in n_range]
powers_medium = [power_analysis.solve_power(effect_size=0.05, alpha=0.05, nobs=n, alternative="two-sided") for n in n_range]

axes[1].plot(n_range, powers_small,  label="MDE=0.02 (tiny)")
axes[1].plot(n_range, powers_medium, label="MDE=0.05 (small)")
axes[1].axhline(0.80, color="black", linestyle=":", label="80% power target")
axes[1].set_xlabel("Test-set size")
axes[1].set_ylabel("Power")
axes[1].set_title("Power vs test-set size for different MDEs")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{base}/05_significance.png", dpi=120)
plt.show()

In [ ]:
def practical_significance_report(delta_auc, n_daily_predictions, revenue_per_correct, alpha=0.05, test_n=600):
    """
    Convert ΔAUC to estimated daily business impact.
    Very rough framing: treats ΔAUC as proxy for improved precision on a balanced dataset.
    """
    extra_correct_per_day = int(delta_auc * n_daily_predictions)
    daily_revenue_impact  = extra_correct_per_day * revenue_per_correct

    power_obj = TTestPower()
    effect_size = cohens_h(0.83 + delta_auc, 0.83)
    power = power_obj.solve_power(effect_size=max(effect_size, 0.001), alpha=alpha, nobs=test_n, alternative="two-sided")

    return {
        "delta_auc"              : delta_auc,
        "extra_correct_per_day"  : extra_correct_per_day,
        "daily_revenue_impact"   : f"${daily_revenue_impact:,.0f}",
        "test_n"                 : test_n,
        "statistical_power"      : f"{power:.0%}",
    }

report = practical_significance_report(
    delta_auc=0.015,
    n_daily_predictions=50_000,
    revenue_per_correct=2.50,
    test_n=600
)
for k, v in report.items():
    print(f"  {k:<30}: {v}")